EDA - Analyzing and understanding dataset (9 dfs) by summarizing its structure, checking data quality, exploring distributions, and identifying patterns before applying any transformation or advanced analysis.

Orders DF Analysis

In [0]:
orders = spark.read.csv('/Volumes/workspace/default/olist_dataset/orders.csv',header = True)
display(orders)

In [0]:
print(orders.count(),len(orders.columns))
print("Columns:",orders.columns)

In [0]:
orders.printSchema()

In [0]:
from pyspark.sql.functions import *
orders.select([sum(when((col(c).isNull()),1).otherwise(0)).alias(c) for c in orders.columns]).show()

In [0]:
orders.count() - orders.distinct().count()

In [0]:
orders.groupBy('customer_id').count().filter('count > 1').show()

In [0]:
orders.groupBy('order_status').count().show()

In [0]:
selected_cols = ['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date']
orders.groupBy('order_status').agg(
    *[sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in selected_cols]
    ).show() 

Customer DF Analysis

In [0]:
customers = spark.read.csv('/Volumes/workspace/default/olist_dataset/customers.csv',header = True)
display(customers)

In [0]:
print(customers.count(), len(customers.columns))
print("Columns:",customers.columns)

In [0]:
customers.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in customers.columns]).show()

In [0]:
customers.count() - customers.distinct().count()

In [0]:
customers.select('customer_city').distinct().count(), customers.select('customer_state').distinct().count()

Order Items DF Analysis

In [0]:
order_items = spark.read.csv('/Volumes/workspace/default/olist_dataset/order_items.csv',header = True)
display(order_items)

In [0]:
print(order_items.count(),len(order_items.columns))
print("Columns:",order_items.columns)

In [0]:
order_items.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in order_items.columns]).show()

In [0]:
order_items.groupBy('product_id').count().filter('count > 1').show()

In [0]:
order_items.count() - order_items.distinct().count()

In [0]:
order_items.groupBy('order_id').count().filter('count > 1').show(truncate=False)

In [0]:
order_items.groupBy('order_id','order_item_id').count().filter('count > 1').show(truncate=False)

In [0]:
order_items = order_items.withColumn('price',col('price').cast('double'))
order_items = order_items.withColumn('freight_value',col('freight_value').cast('double'))

In [0]:
order_items.select('price','freight_value').summary().show()

In [0]:
#Finding Outliers in Price
Q1, Q3 = order_items.approxQuantile('price', [0.25, 0.75], 0.0)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
order_items.filter((col("price") < lower_bound) | (col("price") > upper_bound)).count()

In [0]:
#Finding Outliers in Freight value
Q1, Q3 = order_items.approxQuantile('freight_value', [0.25, 0.75], 0.0)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
order_items.filter((col("freight_value") < lower_bound) | (col("freight_value") > upper_bound)).count()

In [0]:
order_items.groupBy('price').count().orderBy("price",ascending = False).limit(20).show()

Products DF Analysis

In [0]:
products = spark.read.csv('/Volumes/workspace/default/olist_dataset/products.csv',header = True)
display(products)

In [0]:
print(products.count(), len(products.columns))
print("Columns:",products.columns)

In [0]:
products.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in products.columns]).show()

In [0]:
#Duplicate Rows
products.count() - products.distinct().count()

In [0]:
products.select('product_category_name').distinct().count()

In [0]:
products = products.withColumn('product_weight_g',col('product_weight_g').cast('double'))
products = products.withColumn('product_length_cm',col('product_length_cm').cast('double'))
products = products.withColumn('product_height_cm',col('product_height_cm').cast('double'))
products = products.withColumn('product_width_cm',col('product_width_cm').cast('double'))

In [0]:
products.select('product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm').summary().show()

In [0]:
#Oultiers 
Q1,Q3 = products.approxQuantile('product_weight_g',[0.25,0.75],0.0)
IQR = Q3-Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
products.filter((col("product_weight_g") < lower_bound) | (col("product_weight_g") > upper_bound)).count()

In [0]:
Q1, Q3 = 18.0, 38.0
IQR = Q3-Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
products.filter((col("product_length_cm") < lower_bound) | (col("product_length_cm") > upper_bound)).count()

Product Category Translation DF Analysis

In [0]:
pcat_trans = spark.read.csv('/Volumes/workspace/default/olist_dataset/product_category_name_translation.csv',header = True,quote='"',
    escape='"', multiLine= True)
display(pcat_trans)

In [0]:
print(pcat_trans.count(), len(pcat_trans.columns))
print("Columns:",pcat_trans.columns)

In [0]:
pcat_trans.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in pcat_trans.columns]).show()

In [0]:
#duplicate rows
pcat_trans.select('product_category_name').distinct().count()

In [0]:
products.groupBy('product_category_name').count().join(pcat_trans.select('product_category_name'),on='product_category_name',how='left_anti').show(truncate=False)

Seller DF Analysis

In [0]:
seller = spark.read.csv('/Volumes/workspace/default/olist_dataset/sellers.csv',header = True)
display(seller)

In [0]:
print(seller.count(), len(seller.columns))
print("Columns:",seller.columns)

In [0]:
seller.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in seller.columns]).show()

In [0]:
#Duplicate Rows
seller.count() - seller.distinct().count()

In [0]:
seller.select('seller_city').distinct().count(), seller.select('seller_state').distinct().count()

Geolocation DF analysis

In [0]:
location = spark.read.csv('/Volumes/workspace/default/olist_dataset/geolocation.csv',header = True)
display(location)

In [0]:
print(location.count(), len(location.columns))
print("Columns:",location.columns)

In [0]:
location.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in location.columns]).show()

In [0]:
#Duplicate Rows
location.count() - location.distinct().count()

In [0]:
location.select('geolocation_city').distinct().count(), location.select('geolocation_state').distinct().count()

Order Payments DF Analysis

In [0]:
payments = spark.read.csv('/Volumes/workspace/default/olist_dataset/order_payments.csv',header = True)
display(payments)

In [0]:
print(payments.count(),len(payments.columns))
print("Columns:",payments.columns)

In [0]:
payments.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in payments.columns]).show()

In [0]:
#duplicate records
records - payments.distinct().count()

In [0]:
payments.groupBy('order_id').count().filter('count > 1').show()

In [0]:
payments = payments.withColumn('payment_value',col('payment_value').cast('double'))\
    .withColumn('payment_installments',col('payment_installments').cast('int'))\
    .withColumn('payment_sequential',col('payment_sequential').cast('int'))

In [0]:
payments.summary().show()

In [0]:
payments.groupBy('payment_type').count().show()

Order Review DF Analysis

In [0]:
reviews = spark.read.csv('/Volumes/workspace/default/olist_dataset/order_reviews.csv',header = True,quote='"',
    escape='"', multiLine= True)
display(reviews)

In [0]:
print(reviews.count(), len(reviews.columns))
print("COlumns:",reviews.columns)

In [0]:
reviews.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in reviews.columns]).show()

In [0]:
reviews.count() - reviews.distinct().count()

In [0]:
#to check 1 order has how many reviews - max is 3 and min in 1
reviews.groupBy('order_id').count().filter('count > 1').orderBy('count',ascending = False).show()

In [0]:
reviews.select("review_score").distinct().show()

In [0]:
reviews.select('review_score').describe().show()

In [0]:
reviews.select('review_comment_title').distinct().count()

Exploratory Data Analysis: Olist E-commerce Dataset
1. Orders DF
-   The orders dataset contains 99,441 records and 8 columns, capturing the lifecycle of each order from purchase to delivery.
-   No duplicate records were found.
- Missing values are observed in delivery-related timestamps:
  - order_approved_at: 160 nulls
  - order_delivered_carrier_date: 1,783 nulls
  - order_delivered_customer_date: 2,965 nulls
- These nulls are expected and can be linked to order status such as canceled or unavailable.
- Order Status Distribution
Majority of orders are delivered (96,478)
  - Small proportions fall into categories like:
  - canceled (625)
  - unavailable (609)
  - shipped, processing, invoiced (minor counts)\
The dataset is heavily skewed toward successfully delivered orders, indicating overall operational efficiency.

2. Customers DF
- Contains 99,441 rows and 5 columns
- No missing values or duplicates
- Categorical Analysis:
  - 4,119 unique cities and 27 states

3. Order Items DF
- Contains 112,650 rows and 7 columns
- No null values or duplicates
- Price Analysis:
  - Range: 0.85 to 6,753
  - Mean: 120.65
  - Right skewed (mean > median)
  - 8,427 outliers, mostly on the higher end
  - Large gap between Q3 (134.9) and max (6,753)\
Most products are low-priced, but a small number of high-value items significantly influence the distribution.
- Freight Value Analysis:
  - Range: 0 to 409.68
  - Mean: 19.99
  - 12,134 outliers
  - Slightly right skewed
  - Lower standard deviation compared to price\
Freight costs are relatively stable and consistent, with only a few extreme values.

4. Products DF
- Contains 32,951 rows and 9 columns
- Missing values:
  - 610 nulls across category and description-related fields
  - 2 nulls in physical attributes (weight & dimensions)
- Total 74 unique product categories
- Product Weight Analysis
  - Minimum value = 0 (invalid)
  - Very high standard deviation → high variability
  - 4451 i.e approx 13% outliers
  - Right skewed (mean > median)\
Product weights vary significantly, with most items being lightweight and a small number of extremely heavy products.

5. Product Category Translation
- Contains 71 categories, while products dataframe has 74 unique categories
- Missing categories:(which are in products DF and not here )
  - pc_gamer
  - portateis_cozinha_e_preparadores_de_alimentos
  - NULL category (610 records)\
There are inconsistencies between product categories and their English translations, requiring data cleaning.

6. Sellers DF
- 3,095 rows and 4 columns
- No missing values or duplicates
- Categorical Analysis:
  - 611 cities
  - 23 states\
Seller distribution is moderately diverse but less widespread than customers.

7. Geolocation DF
- Contains 1,000,163 rows
- No null values
- 261,831 duplicate rows
- It has:
  - 8,011 cities and 27 unique states\
High duplication suggests multiple entries per zip code, likely due to granularity in geographic data.

8. Payments DF
- 103,886 rows and 5 columns
- No nulls or duplicates
- Payment Types( Categorical Analysis):
  - Credit Card: 76,795
  - Boleto: 19,784
  - Voucher: 5,775
  - Debit Card: 1,529\
Credit cards dominate transactions
Boleto (bank slip payment) is widely used
Some orders involve multiple payment records
- Payment Behavior:
  - payment_sequential shows that some orders are split into multiple transactions
  - payment_installments indicates EMI usage

9. Reviews DF
- 99,224 rows and 7 columns
- No duplicates
- Missing values:
  review_comment_title: 87,656
  review_comment_message: 58,247
- Review scores are in between 1 to 5